In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
df = pd.read_csv("../data/arxiv_cleaned.csv")

In [3]:
df.shape

(41127, 6)

In [4]:
df.columns

Index(['terms', 'titles', 'abstracts', 'title length', 'abstract length',
       'text'],
      dtype='str')

In [4]:
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=50000
)

In [5]:
tfidf_matrix=tfidf.fit_transform(df['text'])

In [6]:
from scipy.sparse import save_npz

save_npz(
    "../models/tfidf_matrix.npz",
    tfidf_matrix
)

In [7]:
tfidf_matrix.shape

(41127, 50000)

In [8]:
similarity_score=cosine_similarity(tfidf_matrix[0],
                                   tfidf_matrix)

In [9]:
similarity_score.shape

(1, 41127)

In [11]:
similar_indices=similarity_score.flatten().argsort()[::-1]

In [12]:
similar_indices[:10]

array([    0,   372,    74,   301, 13023, 21464, 36058, 39202, 14064,
       40426])

In [13]:
similar_indices=similar_indices[similar_indices != 0]

In [14]:
top_indices = similar_indices[:5]
top_indices

array([  372,    74,   301, 13023, 21464])

In [15]:
df.iloc[top_indices][['titles']]

,titles
372,Graph Convolutional Networks with EigenPooling
74,Graph-MLP: Node Classification without Message...
301,Hierarchical Graph Pooling with Structure Lear...
13023,Edge but not Least: Cross-View Graph Pooling
21464,Accurate Learning of Graph Representations wit...


In [16]:
df.iloc[0][['titles', 'abstracts']]

titles       Multi-Level Attention Pooling for Graph Neural...
abstracts    Graph neural networks (GNNs) have been widely ...
Name: 0, dtype: object

In [8]:
def recommend_papers(paper_index, top_k=5):
    
    similarity_scores = cosine_similarity(
        tfidf_matrix[paper_index],
        tfidf_matrix
    ).flatten()

    similar_indices = similarity_scores.argsort()[::-1]

    similar_indices = similar_indices[similar_indices != paper_index]

    top_indices = similar_indices[:top_k]

    recommendations = df.iloc[top_indices][['titles', 'abstracts']].copy()
    
    recommendations['similarity_score'] = similarity_scores[top_indices]

    return recommendations


In [21]:
recommend_papers(0, 5)

,titles,abstracts,similarity_score
372,Graph Convolutional Networks with EigenPooling,"Graph neural networks, which generalize deep n...",0.449377
74,Graph-MLP: Node Classification without Message...,Graph Neural Network (GNN) has been demonstrat...,0.397364
301,Hierarchical Graph Pooling with Structure Lear...,"Graph Neural Networks (GNNs), which generalize...",0.397326
13023,Edge but not Least: Cross-View Graph Pooling,Graph neural networks have emerged as a powerf...,0.393036
21464,Accurate Learning of Graph Representations wit...,Graph neural networks have been widely used on...,0.374575


In [22]:
recommend_papers(100, top_k=5)

,titles,abstracts,similarity_score
37434,Tree Tensor Networks for Generative Modeling,"Matrix product states (MPS), a tensor network ...",0.418450
3972,A Multi-Scale Tensor Network Architecture for ...,We present an algorithm for supervised learnin...,0.345045
6258,A Microscopic Pandemic Simulator for Pandemic ...,Microscopic epidemic models are powerful tools...,0.321397
22194,Multi-layered tensor networks for image classi...,The recently introduced locally orderless tens...,0.319794
11656,Patch-based medical image segmentation using Q...,Tensor networks are efficient factorisations o...,0.302009
